# U7-L3：多线程协作

**课程目标**

1. 能解释视觉处理与语音播放的等待关系，用一个工作线程让检测继续更新。
2. 能用同一把锁交接完整检测记录，区分结果处理耗时、处理后存放时长与源时间戳。
3. 能发出停止请求并核查线程与音频任务是否结束，在未结束时保留原对象、不重复启动。

## 一、任务：播报时继续观察目标

先想象按顺序写程序：取图检测一次，播放一段话，等待播放结束，再取下一张图。播放期间同学把球移了位置，程序却要等说完才知道。若下一步需要根据目标变化作判断，只保存播报前的检测结果就不够。

本课把两份工作分开：一份不断取图、检测、保存最近记录，另一份发起有限播报并查看记录。机器人保持静止，只由同学缓慢移动目标。先掌握这种执行安排，不同时加入接近或踢球运动。

### 1.1 线程怎样改变等待关系

**线程**是程序中一条执行路径。Notebook当前正在执行的代码称为主线程；再创建一个检测工作线程后，主线程等待音频时，检测函数仍有机会继续运行。线程不保证每步同时完成，也不会自动提高相机或推理速度。

`threading.Thread(target=detect_loop)`创建对象，并指定要执行哪个函数；`start()`才开始执行这个函数。这里传函数名，不写`detect_loop()`，否则会先在当前代码中直接调用它。

| 执行位置 | 播报期间做什么 | 结果交给谁 |
| --- | --- | --- |
| 检测工作线程 | 反复取图、检测、保存编号与坐标 | 主线程稍后读取最近一份记录 |
| 主线程 | 发起合成与播放、等待结果、读取前后记录 | 操作者比较本次实验 |

### 1.2 与L1订阅回调有什么联系

L1把回调函数交给SDK，SDK收到数据时调用它。本课自己创建线程，在线程函数里反复调用取图与检测。两种写法都能让某些工作在当前等待期间继续，但本课需要主动管理自己创建的线程：启动之后，结束时要请求停止并检查它是否真的退出。

实验用两份记录比较：播报前记`before`，播报后记`after`。如果编号增加，说明期间又完成了检测；再看是否检出和坐标变化，判断得到的记录具体是什么。

## 二、准备检测与播报

⚠️ **实验安全**：机器人保持固定姿态，不发运动指令；同学只移动球，不碰机器人。音量适中，在踢球实验之外的安静位置进行。

📌 **前置**：L2管理器已`stop()`；其他Notebook的线程、订阅、识别、chat及运动任务均已结束。`x_lab`文件夹与本课Notebook并排。

本课使用两个课程附件函数：`select_ball()`按类别选择评分最高的目标；`say()`把U5“合成有限文本—取得音频—播放并等待”的代码放在一起。`new_session()`只建立一个字典，保存机器人、语音对象和音频/运动任务句柄；它不是SDK的聊天会话，也不会调用`chat()`。

**STEP 01：首次运行连接格。** 核对模式，保留`session`和`vision_thread`以便之后收尾。重复实验从STEP 02开始，不重跑本格覆盖仍在使用的对象。默认检测`Ball`；若改成来宾，检测模型和类别必须一起修改。

In [ ]:
import time
import threading
from boosteros.robots.booster import BoosterRobot
from boosteros.brain import Detection, Speech
from x_lab.ball_position import select_ball
from x_lab.flows import new_session, say, idle, cleanup

robot = BoosterRobot()
speech = Speech(robot)
detector = Detection(model="soccer")
TARGET_CLASS = "Ball"  # 也可统一改模型与类别，用person观察来宾
vision_thread = None
session = new_session(robot, speech)
print("模式：", robot.get_mode())

### 2.1 怎样交接一整份记录

检测线程要写编号、是否检出、坐标和时间，主线程要读这些值。如果写到一半就被读取，可能看到“新编号、旧坐标”的组合。为让双方按次序使用共享记录，我们创建一把**锁**，读和写都先取得这同一把锁，用完后归还。

`with lock:`表示取得锁后执行下面缩进的代码，离开时释放锁。另一方需要同一把锁时先等待。这里仅在锁内更新或复制字典；取图、推理、播放都放在外面，否则一方耗时较长，另一方就一直读写不了。

### 2.2 怎样通知检测线程结束

`STOP = threading.Event()`建立停止标志，开始未置位。主线程调用`STOP.set()`发出停止请求；检测循环在`STOP.is_set()`处看见请求后退出。它不会强行打断正在进行的取图或检测，因此请求发出后还要等待并检查。

**STEP 02：先确认旧线程和音频已结束，再运行下一格。** 它建立本轮锁、停止标志和初始共享字典，不启动线程。看到“旧线程仍在运行”或“旧音频任务未结束”时先做STEP 07；不能删除检查或重建STOP来掩盖旧线程。

In [ ]:
if vision_thread is not None and vision_thread.is_alive():
    raise RuntimeError("旧检测线程仍在运行，先执行STEP 07")
if not idle(session):
    raise RuntimeError("旧音频任务未结束，先执行STEP 07")

lock = threading.Lock()
STOP = threading.Event()
shared = {"number": 0, "found": False, "u": None, "v": None,
          "received": None, "finished": None, "source": None, "error": None}

**STEP 03：定义检测和读取函数，暂不启动。** `detect_loop()`反复取图、检测，得到结果后在锁内更新记录；未检出时把`u/v`写为`None`，不沿用上一轮的位置。异常信息存入`error`，并发出停止请求。`read_observation()`使用同一把锁复制字典，供主线程查看。

| 字段 | 何时记录/表示什么 | 怎样使用 |
| --- | --- | --- |
| `number` | 每完成一次检测加1 | 比较播报期间完成了几次 |
| `found`、`u/v` | 是否检出、框中心像素 | 未检出不解释坐标，也不把空值当画面中心 |
| `received` | 本轮调用取图前的本地时刻 | 本课字段沿用此名，尚不是源采集时刻 |
| `finished` | 本轮推理结束的本地时刻 | 减`received`得本次取图与处理耗时 |
| `source` | 图像自带源时间戳 | 保存来源，不与本地时钟直接相减 |
| `error` | 工作函数捕获的异常 | 说明为何停止，保留待查内容 |

注意：L1的`received`在回调开始时记，本课在取图前记。相同字段名必须结合记录位置理解；本课随后说的“处理耗时”和“处理后存放时长”分别有明确起止事件。

In [ ]:
def detect_loop():
    try:
        while not STOP.is_set():
            received = time.monotonic()
            image = robot.get_image(img_type="rgb")
            target = select_ball(detector.detect(image.to_numpy()), TARGET_CLASS)
            u, v = None, None
            if target is not None:
                u, v = target.bbox.center_x, target.bbox.center_y
            finished = time.monotonic()
            with lock:
                shared.update(number=shared["number"]+1, found=target is not None,
                              u=u, v=v, received=received, finished=finished,
                              source=image.header.timestamp)
            time.sleep(0.1)
    except Exception as exc:
        with lock:
            shared["error"] = str(exc)
        STOP.set()

def read_observation():
    with lock:
        return dict(shared)

## 三、比较播报前后的检测

**STEP 04：运行一整轮，按输出提示移动目标。** 下一格先创建并启动检测线程，等约一秒取`before`，随后播报，播报返回后取`after`。记录员看前后编号；另一名同学在听到播报时缓慢把球从画面一侧移向另一侧，不碰机身。

`say(session, 文本)`是课程函数：内部调用`synthesize()`，成功后用`result()`取得音频，再调用`play_sound()`并等待。它把句柄存进`session`，只在相应任务成功结束时返回“完成”；超时会请求取消并返回原因。打开`x_lab/flows.py`可找到这两个SDK调用。

启动、比较、停止放在同一格：`try`内做实验，`finally`内无论正常结束还是抛出异常，都先`STOP.set()`，再`join(timeout=3.0)`等待线程退出。`join`最多等待三秒，是否真的结束由`is_alive()`判断。打印仍在运行时，先执行STEP 07，不能再启动一轮。

In [ ]:
before = None
after = None
if STOP.is_set() or (vision_thread is not None and vision_thread.is_alive()):
    raise RuntimeError("请先收尾，再从STEP 02开始")
vision_thread = threading.Thread(target=detect_loop, daemon=True)
try:
    vision_thread.start()
    time.sleep(1.0)
    before = read_observation()
    print("播报前：", before)
    result = say(session, "我正在介绍这个场景。请同学缓慢移动目标，观察检测是否仍在更新。")
    after = read_observation()
    print("播报结果：", result)
    print("播报后：", after)
finally:
    STOP.set()
    vision_thread.join(timeout=3.0)
print("检测线程仍在运行：", vision_thread.is_alive())

### 3.1 记录增加了，是否就能一直使用

**STEP 05：运行下一格，解释两个时长。** `after`是播报返回时复制下来的字典，后续不会自动变化。

- `finished - received`：这次调用取图到检测完成用了多久。
- `现在 - finished`：检测完成后，这份记录又放了多久。

两段合起来，是从本轮取图前到现在的本地经过时长，仍没有证实图像最初采集到取图之间等了多久。下一格以“检出了目标，且这个本地经过时长不超过一秒”作练习判断；一秒是待本机核对的参数，不是运动安全标准。即使显示`True`，还要核对检测内容是否符合实际。

编号增加说明又完成了几次检测；它不证明目标选对，也不证明图像采集时刻同步。把输出与球实际移动方向一起记下来。

In [ ]:
if before is None or after is None or after["received"] is None:
    print("本次没有完整比较记录")
else:
    print("期间完成检测次数：", after["number"]-before["number"])
    processing = after["finished"]-after["received"]
    stored = time.monotonic()-after["finished"]
    print(f"本次取图和处理{processing:.3f}秒，处理后又经过{stored:.3f}秒")
    usable = after["found"] and time.monotonic()-after["received"] <= 1.0
    print("按本次条件是否仍可使用：", usable)

**STEP 06：等约两秒，再运行STEP 05。** 此时STEP 04已经请求停止线程，`after`还是那份旧字典。比较“处理后又经过”是否增加，以及判断是否从`True`变为`False`；若最初已是`False`，说明它原本就不满足本次条件，不必改结果。

### 3.2 退出后怎样重新开始

**STEP 07：运行下一格，分别检查检测线程和音频任务。** STOP请求、线程退出、SDK音频结束是三件相关但不同的事。`join()`返回后看`is_alive()`，音频句柄则看自己的终态；线程结束不会自动取消SDK任务。

全部结束且无待查异常后，重复一轮按STEP 02重新初始化，STEP 03重定义函数，再运行STEP 04～07。旧线程未结束时保留原STOP、锁和线程对象，交由教师处理；不要用再次运行连接格的方式“清空”。

In [ ]:
STOP.set()
if vision_thread is not None:
    vision_thread.join(timeout=3.0)
    print("线程仍在运行：", vision_thread.is_alive())
print("音频待处理项：", cleanup(session))
print("音频任务均已结束：", idle(session))
print("检测错误：", read_observation()["error"])

本课不运动，退出后按教师安排保持安全姿态或执行人工关机流程。

## 四、记录与学习检查

| 轮次 | 播报前/后编号 | 目标和像素坐标变化 | 取图处理/处理后存放时长 | 播报返回原因 | 线程/音频退出 |
| --- | --- | --- | --- | --- | --- |
| 第1轮 | | | | | |
| 第2轮 | | | | | |

### 思考题

1. 主线程等待播放时，检测编号为什么还能增加？指出启动工作线程的那行。
2. 找到更新记录和读取记录的两处`with lock:`。若把取图也放在锁里，另一方会受到什么影响？
3. STOP已置位而`is_alive()`仍为真，说明什么？这时为何要保留原对象？
4. 重跑STEP 05时记录没有新检测，为什么“是否仍可使用”可能改变？用两个时长说明。

**完成核对**：保存两轮前后记录及至少一次旧记录复查，能解释字段起止事件，线程和SDK任务分别核查；未结束项如实注明。

### 检查点

- [__] 能从前后记录说明播放等待期间检测是否继续更新。
- [__] 能指出两处锁的作用，区分取图处理耗时、处理后存放时长与源时间戳。
- [__] 能分别核查工作线程与SDK音频任务退出，保留未结束对象，不重复启动。


## 五、学习评价

请独立作答；操作完成情况另按本课记录与检查点核对。

1. 判断：主线程等待音频播放期间，已经启动的检测工作线程仍可能继续更新记录。（　）

2. 填空：更新和读取完整检测记录时使用同一把________；`join()`返回后，还应通过________检查线程是否仍在运行。

3. 选择：停止请求已发出，但检测线程或音频任务仍未结束，应（　）。
   A. 重跑连接格覆盖旧对象
   B. 保留原线程、停止信号和任务句柄，分别核查退出情况
   C. 认为线程退出会自动取消所有SDK任务
   D. 用处理后存放时长代替图像源时间戳

> 参考答案（教师留存，勿印发给学生）：
> 1. √。2. 锁；`is_alive()`。3. B；线程与SDK任务分别检查。
